In [5]:
import os
import numpy as np
import torch
import torch.nn as nn
import mne
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib as mpl
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.animation")
print("🚨 Running Real-Time Simulation.....")


# SETTINGS
mpl.rcParams['animation.embed_limit'] = 50

MODEL_PATH = "cnn_lstm_seizure_state_dict.pth"

DATA_PATH = r"C:\Research Paper\chbmit_data\chb01"
EDF_FILE = "chb01_03.edf"

SFREQ = 128
WINDOW_SEC = 30
HOP_SEC = 5

SEIZURE_START = 2996
SEIZURE_END = 3036

PRE_CONTEXT = 60
POST_CONTEXT = 60


# MODEL
class CNNLSTM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, 32, 3)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, 3)
        self.pool2 = nn.MaxPool1d(2)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.fc1 = nn.Linear(64, 64)
        self.fc2 = nn.Linear(64, 1)

    def _forward_features(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = x.permute(0, 2, 1)
        _, (h, _) = self.lstm(x)
        x = torch.relu(self.fc1(h[-1]))
        x = torch.sigmoid(self.fc2(x))
        return x.squeeze()


# LOAD DATA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw = mne.io.read_raw_edf(os.path.join(DATA_PATH, EDF_FILE), preload=True, verbose="ERROR")
raw.resample(SFREQ)

data = raw.get_data().astype(np.float32)

data = data[:, int((SEIZURE_START - PRE_CONTEXT) * SFREQ):
               int((SEIZURE_END + POST_CONTEXT) * SFREQ)]

channels = data.shape[0]


# LOAD MODEL
model = CNNLSTM(channels).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()


# NORMALIZE
def normalize(x):
    return (x - x.mean()) / (x.std() + 1e-8)


# WINDOW SETTINGS
window_size = SFREQ * WINDOW_SEC
hop_size = SFREQ * HOP_SEC
total_samples = data.shape[1]
max_frames = (total_samples - window_size) // hop_size


# STORAGE FOR TABLE
records = []


# PLOT
fig, ax = plt.subplots(figsize=(10,5))
line, = ax.plot([], [], lw=2)

ax.axhline(0.3, linestyle='--')

ax.set_ylim(0, 1)
ax.set_xlim(0, 60)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Seizure Probability")

title = ax.set_title("Real-Time Seizure Detection")

time_points = []
pred_probs = []


# UPDATE FUNCTION
def update(frame):
    start = frame * hop_size
    end = start + window_size

    window = normalize(data[:, start:end])
    window_tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        prob = model(window_tensor).item()

    time_sec = start / SFREQ
    status = "Seizure" if prob > 0.3 else "Normal"

    records.append([time_sec, status, prob])

    time_points.append(time_sec)
    pred_probs.append(prob)

    line.set_data(time_points, pred_probs)

    if prob > 0.3:
        ax.set_facecolor("#ffcccc")
        title.set_text(f"Seizure Detected | Time={time_sec:.1f}s | Prob={prob:.2f}")
    else:
        ax.set_facecolor("white")
        title.set_text(f"Normal EEG | Time={time_sec:.1f}s | Prob={prob:.2f}")

    ax.set_xlim(0, max(60, time_sec + 5))

    return line,


# ANIMATION
ani = FuncAnimation(fig, update, frames=max_frames, interval=600, repeat=False)

plt.tight_layout()
plt.close(fig)


# DISPLAY GRAPH
from IPython.display import HTML

global_anim = ani
display(HTML(ani.to_jshtml()))

print("\n\n\n\n\n")


# PROFESSIONAL TABLE STYLING
df = pd.DataFrame(records, columns=["Time (s)", "Status", "Probability"])

df["Time (s)"] = pd.to_numeric(df["Time (s)"], errors='coerce').round(1)
df["Probability"] = pd.to_numeric(df["Probability"], errors='coerce').round(3)

def highlight_rows(row):
    if row["Status"] == "Seizure":
        return ['font-weight: bold; color: #b22222'] * len(row) 
    else:
        return [''] * len(row)

styled_df = (
    df.style
    .apply(highlight_rows, axis=1)
    .set_properties(**{
        'text-align': 'center',
        'font-size': '12pt',
        'border': '1px solid #ddd'
    })
    .set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#2f2f2f'),
            ('color', 'white'),
            ('font-size', '13pt'),
            ('text-align', 'center'),
            ('border', '1px solid #ddd')
        ]},

        {'selector': 'td', 'props': [
            ('border', '1px solid #ddd')
        ]},

        {'selector': 'tbody th', 'props': [
            ('background-color', '#fafafa'),
            ('color', '#333'),
            ('border', '1px solid #ddd'),
            ('font-weight', 'normal')
        ]}
    ])
)

styled_df = styled_df.format({
    "Time (s)": "{:.1f}",
    "Probability": "{:.3f}"
})

display(styled_df)

🚨 Running Real-Time Simulation.....


,Time (s),Status,Probability
0,0.0,Normal,0.002
1,0.0,Normal,0.002
2,5.0,Normal,0.002
3,10.0,Normal,0.003
4,15.0,Normal,0.007
5,20.0,Normal,0.001
6,25.0,Normal,0.005
7,30.0,Normal,0.010
8,35.0,Seizure,0.566
9,40.0,Seizure,0.995
